# qust 基础用法

这本 notebook 从最基础的表达式开始，系统展示 qust 的常用写法：

- `col(...)` 如何选择列、构造 literal、组合表达式；
- `select / with_cols / filter / alias / cast / fill / drop_cols` 等常用算子；
- 行算子、横向算子、rolling/expanding/group_by/over 上下文；
- 如何直接嵌入 Polars 表达式；
- `calc_data`、`runtime`、`calc_stream` 的区别；
- 数据源：DataFrame、Parquet、Polars LazyFrame、ClickHouse。

所有本地示例都使用 `/root/qust-py/examples/data/data_kline2.parquet`。


In [65]:
import sys
sys.path.insert(0, "/root/otters/otters-py/python")

import os
import importlib

import qust as qs
qs = importlib.reload(qs)

from qust import col, pms
from qust._polars import pl
import qust.datasource as qds

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(14)

DATA_KLINE = "/root/qust-py/examples/data/data_kline2.parquet"
DATA_FUTURE = "/root/qust-py/examples/data/kline_data_all.parquet"
DATA_STOCK = "/root/qust-py/examples/data/stock_data_kline.parquet"


## 1. 读取样例数据

这里先用 Polars 读取一份小样本。qust 的 `calc_data(data)` 可以直接接收 `pl.DataFrame`。


In [66]:
raw = pl.read_parquet(DATA_KLINE)
data = raw.filter(pl.col("ticker") == "au").head(1_000)

print("raw shape:", raw.shape)
print("sample shape:", data.shape)
data.head(5)


raw shape: (610463, 8)
sample shape: (1000, 8)


ticker,datetime,open,high,low,close,volume,is_finished
str,datetime[ms],f64,f64,f64,f64,f64,bool
"""au""",2022-07-02 00:01:00,390.160004,390.160004,390.059998,390.119995,81.0,true
"""au""",2022-07-02 00:02:00.500,390.119995,390.140015,390.079987,390.140015,52.0,true
"""au""",2022-07-02 00:03:00,390.140015,390.200012,390.119995,390.200012,50.0,true
"""au""",2022-07-02 00:04:01,390.200012,390.220001,390.140015,390.160004,61.0,true
"""au""",2022-07-02 00:05:00.500,390.140015,390.140015,390.079987,390.100006,41.0,true


## 2. `col(...)` 是表达式入口

常见入口：

- `col("close")`：选择单列；
- `col("open", "close")`：选择多列；
- `col.all`：选择所有列；
- `col.lit(1.0)`：literal；
- `col(0)`：选择当前表达式输出的第 0 列；
- `col.from_df(df)`：把小 DataFrame 内嵌成表达式；
- `col.series(...)`：把 Series 内嵌成表达式。


In [67]:
examples = {
    "single_col": col("close").calc_data(data).head(3),
    "multi_col": col("open", "close").calc_data(data).head(3),
    "literal": col(col.lit(1.0).alias("one")).calc_data(data).head(3),
    "embedded_df": col.from_df(pl.DataFrame({"name": ["a", "b"], "weight": [0.4, 0.6]})).calc_data(data),
}

for name, out in examples.items():
    print("\n==", name)
    display(out)



== single_col


close
f64
390.119995
390.140015
390.200012



== multi_col


open,close
f64,f64
390.160004,390.119995
390.119995,390.140015
390.140015,390.200012



== literal


one
f64
1.0



== embedded_df


name,weight
str,f64
"""a""",0.4
"""b""",0.6


## 3. `select`、`with_cols`、`filter`

- `select(...)`：只返回指定表达式结果；
- `with_cols(...)`：保留当前输入并追加/覆盖列；
- `filter(...)`：按 predicate 过滤；
- `alias(...)`：改输出列名。

`col.with_cols(...)` 是 `col.all.with_cols(...)` 的短写。


In [68]:
feature_expr = (
    col
    .with_cols(
        ((col("close") - col("open")) / col("open")).alias("bar_ret"),
        (col("high") - col("low")).alias("range"),
    )
    .filter((col("bar_ret") > col.lit(0.0)) & (col("volume") > col.lit(50.0)))
    .select("datetime", "open", "close", "bar_ret", "range")
)

feature_expr.calc_data(data).head(8)


datetime,open,close,bar_ret,range
datetime[ms],f64,f64,f64,f64
2022-07-02 00:02:00.500,390.119995,390.140015,0.000051,0.060028
2022-07-02 00:06:00.500,390.079987,390.140015,0.000154,0.120026
2022-07-02 00:08:03,389.859985,389.880005,0.000051,0.080017
2022-07-02 00:10:02.500,389.880005,389.899994,0.000051,0.079987
2022-07-02 00:12:00.500,389.839996,389.859985,0.000051,0.059998
2022-07-02 00:14:01.500,389.660004,389.720001,0.000154,0.140015
2022-07-02 00:18:00,389.519989,389.559998,0.000103,0.100006
2022-07-02 00:22:00.500,389.179993,389.320007,0.00036,0.160004


## 4. 算术、比较、布尔、缺失值、类型转换

qust 表达式支持 Python 运算符：`+ - * / **`、`> >= < <= == !=`、`& | ~`。常用清洗算子包括 `fill_null`、`fill_nan`、`clip`、`cast`、`is_between` 等。


In [69]:
clean_expr = col(
    (col("close") / col("open") - col.lit(1.0)).alias("ret"),
    (col("close") > col("open")).alias("is_up"),
    col("volume").fill_null(col.lit(0.0)).cast(pl.Float64).alias("volume_f64"),
    col("close").clip(300.0, 500.0).alias("close_clip"),
    col("datetime").dt.date().alias("date"),
)

clean_expr.calc_data(data).head(8)


ret,is_up,volume_f64,close_clip,date
f64,bool,f64,f64,date
-0.000103,false,81.0,390.119995,2022-07-02
0.000051,true,52.0,390.140015,2022-07-02
0.000154,true,50.0,390.200012,2022-07-02
-0.000103,false,61.0,390.160004,2022-07-02
-0.000103,false,41.0,390.100006,2022-07-02
0.000154,true,73.0,390.140015,2022-07-02
-0.000615,false,172.0,389.880005,2022-07-02
0.000051,true,118.0,389.880005,2022-07-02


## 5. 行算子与横向算子

`mean/sum/std/var/min/max/count/rank/first_value/last_value` 等在 qust 里是表达式算子。

- 单列 + 行算子：沿当前输入行聚合，通常输出标量或上下文结果；
- 多列 + `axis=1`：横向按行聚合。


In [70]:
row_expr = col(
    col("close").mean().alias("close_mean"),
    col("close").std().alias("close_std"),
    col("close").rank().alias("last_rank"),
    col("open", "close", "high", "low").mean(axis=1).alias("ohlc_row_mean"),
)

row_expr.calc_data(data)


close_mean,close_std,last_rank,ohlc_row_mean
f64,f64,u32,f64
390.17764,1.137732,12,390.125
390.17764,1.137732,12,390.120003
390.17764,1.137732,12,390.165009
390.17764,1.137732,12,390.180008
390.17764,1.137732,12,390.115005
390.17764,1.137732,12,390.125
390.17764,1.137732,12,390.0
390.17764,1.137732,12,389.884995
…,…,…,…


## 6. 上下文：rolling、expanding、group_by、over

上下文改变行算子的执行范围：

- `.rolling(20)`：每行使用最近 20 行窗口；
- `.expanding()`：从开始到当前行的累计窗口；
- `.group_by("ticker")`：按 key 聚合，输出长度通常变成分组数；
- `.over("ticker")`：按 key 独立维护状态，但输出仍对齐原始行。


In [71]:
context_expr = (
    col
    .with_cols(
        col("close").mean().rolling(20).alias("ma20"),
        col("close").mean().expanding().alias("mean_so_far"),
    )
    .over("ticker")
    .select("ticker", "datetime", "close", "ma20", "mean_so_far")
)

context_expr.calc_data(data).tail(8)


ticker,datetime,close,ma20,mean_so_far
str,datetime[ms],f64,f64,f64
"""au""",2022-07-05 22:18:00,385.119995,385.138,390.215448
"""au""",2022-07-05 22:19:00,385.059998,385.032001,390.210262
"""au""",2022-07-05 22:20:00,385.019989,384.936,390.205045
"""au""",2022-07-05 22:21:00,384.660004,384.861,390.199478
"""au""",2022-07-05 22:22:00,384.480011,384.790001,390.193741
"""au""",2022-07-05 22:23:00,384.899994,384.747,390.188437
"""au""",2022-07-05 22:24:00,384.73999,384.703999,390.182983
"""au""",2022-07-05 22:25:00,384.839996,384.676999,390.17764


In [72]:
grouped = col("close", "volume").mean().group_by("ticker").calc_data(raw.head(50_000))
grouped


ticker,close,volume
str,f64,f64
"""au""",382.341819,165.295141
"""SA""",2504.240207,2650.84054
"""bu""",4131.176406,1576.309124
"""fu""",3162.871648,2389.115913


## 7. 直接调用 Polars 表达式

如果某个操作 Polars 已经写得很好，可以直接把 `pl.Expr` 放进 qust：

- `col(pl.col("close").log().alias("log_close"))`；
- `expr.select(pl.col("x").rank())`。

这种写法适合复用 Polars 生态，但要记住：这是 Polars 表达式语义，不一定有 qust 的流式状态能力。


In [73]:
polars_inside = col(
    pl.col("close").log().alias("log_close"),
    pl.col("close").rank().alias("polars_rank"),
)

polars_inside.calc_data(data.head(8))


log_close,polars_rank
f64,f64
5.966454,4.0
5.966506,5.5
5.966659,8.0
5.966557,7.0
5.966403,3.0
5.966506,5.5
5.965839,1.5
5.965839,1.5


## 8. `display()`、`runtime()`、`calc_data()`

- `expr.display()`：尽量显示成可复制的 Python 表达式；
- `expr.calc_data(data)`：短写，直接编译并执行；
- `expr.runtime()`：先编译成 runtime，适合重复计算或 `plot/calc_stream`。


In [74]:
e = col("close").mean().rolling(20).alias("ma20")
print(e.display())

rt = e.runtime()
print(rt.calc_data(data.head(100)).tail(5))
print(rt.calc_data(data.head(200)).tail(5))


col("close").mean().rolling(20, 20).alias("ma20")
shape: (5, 1)
┌────────────┐
│ ma20       │
│ ---        │
│ f64        │
╞════════════╡
│ 389.099004 │
│ 389.128003 │
│ 389.164003 │
│ 389.199004 │
│ 389.226003 │
└────────────┘
shape: (5, 1)
┌────────────┐
│ ma20       │
│ ---        │
│ f64        │
╞════════════╡
│ 390.270001 │
│ 390.264001 │
│ 390.261002 │
│ 390.258002 │
│ 390.260002 │
└────────────┘


## 9. 数据源：Parquet、DataFrame、LazyFrame

`qds.read_parquet(...)` / `qds.from_dataframe(...)` / `qds.from_lazy(...)` 都可以作为 runtime 的数据源。

- `calc_data(source)`：bounded final 语义，最终返回完整结果；
- `calc_stream(source)`：显式逐批 update 语义，会保留每个 batch 的输出。


In [75]:
parquet_source = qds.read_parquet(DATA_KLINE, n_rows=1_000)
print(col("close").mean().calc_data(parquet_source))

lazy_source = qds.from_lazy(
    pl.scan_parquet(DATA_KLINE).filter(pl.col("ticker") == "au").select("ticker", "datetime", "close").limit(100),
    chunk_size=25,
)
stream_out = col("close").mean().runtime().calc_stream(lazy_source)
stream_out


shape: (1, 1)
┌────────────┐
│ close      │
│ ---        │
│ f64        │
╞════════════╡
│ 2441.93514 │
└────────────┘


close
f64
389.732001
389.632002
389.445335
389.368602


## 10. 从 0 理解 stream：为什么需要 `calc_stream`

很多人一开始会把 qust 当成“另一个 DataFrame API”：给一整张表，算完返回一整张表。这就是 `calc_data(data)`。

但量化里经常不是这样：

- tick 数据一批一批到来，不可能永远等全量数据到了再算；
- rolling/expanding/持仓/订单状态都需要记住历史状态；
- 实盘里每来一个 batch，都要立刻更新信号、监控或风控；
- 数据很大时，一次全读进内存不现实。

这就是 `stream` 的意义：同一个 runtime 保留状态，每来一批数据就 update 一次。

下面用一个 12 行的小表一步一步看清楚。表达式是：

```python
col("v").sum().expanding().alias("cum_v")
```

意思是“从开始到当前行的累计和”。如果 runtime 不保留状态，那么每个 batch 都会从 1 重新累计；如果 runtime 保留状态，第二个 batch 会接着第一个 batch 的结果继续累计。


In [76]:
stream_toy = pl.DataFrame({
    "t": list(range(12)),
    "v": list(range(1, 13)),
})

cum_expr = col("v").sum().expanding().alias("cum_v")

print("输入数据：")
display(stream_toy)

print("一次性 calc_data：")
display(cum_expr.calc_data(stream_toy))


输入数据：


t,v
i64,i64
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9


一次性 calc_data：


cum_v
i64
1
3
6
10
15
21
28
36
45


### 10.1 手动模拟：同一个 runtime 连续接收多个 batch

这里不用 datasource，直接把 DataFrame 切成 3 个 batch。关键是 `rt = cum_expr.runtime()` 只创建一次。

观察输出：

- 第 1 批 `1,2,3,4` 输出 `1,3,6,10`；
- 第 2 批从历史状态 `10` 后继续，输出 `15,21,28,36`；
- 第 3 批继续输出 `45,55,66,78`。

这就是“状态保留”。


In [77]:
rt = cum_expr.runtime()
manual_batches = []

for batch_id, batch in enumerate(stream_toy.iter_slices(4), start=1):
    out = rt.calc_data(batch)
    manual_batches.append(out.with_columns(pl.lit(batch_id).alias("batch_id")))
    print(f"\n输入 batch {batch_id}")
    display(batch)
    print(f"输出 batch {batch_id}")
    display(out)

manual_stream_result = pl.concat(manual_batches)
manual_stream_result



输入 batch 1


t,v
i64,i64
0,1
1,2
2,3
3,4


输出 batch 1


cum_v
i64
1
3
6
10



输入 batch 2


...


输入 batch 3


t,v
i64,i64
8,9
9,10
10,11
11,12


输出 batch 3


cum_v
i64
45
55
66
78


cum_v,batch_id
i64,i32
1,1
3,1
6,1
10,1
15,2
21,2
28,2
36,2
45,3


### 10.2 datasource 版本：`calc_stream(qds.from_dataframe(..., chunk_size=4))`

真实使用时通常不会自己写 for loop，而是把数据源交给 qust：

```python
source = qds.from_dataframe(df, chunk_size=4)
runtime.calc_stream(source)
```

`chunk_size=4` 表示每批 4 行。返回结果和上面的手动循环一致，只是数据读取和 batch 分发由 datasource 负责。


In [78]:
source = qds.from_dataframe(stream_toy, chunk_size=4)
stream_result = cum_expr.runtime().calc_stream(source)
stream_result


cum_v
i64
1
3
6
10
15
21
28
36
45


### 10.3 什么时候用 `calc_data`，什么时候用 `calc_stream`

| 调用 | 适合场景 | 状态语义 | 输出 |
| --- | --- | --- | --- |
| `expr.calc_data(pl_df)` | 小数据、离线一次性研究 | 单次 bounded 计算 | 一张结果表 |
| `expr.runtime().calc_data(batch)` | 自己控制 batch 循环 | runtime 保留状态 | 每个 batch 一张输出 |
| `expr.runtime().calc_stream(source)` | 数据源自动分批、实盘/回放 | runtime 保留状态 | 所有 batch 输出拼接 |
| `qds.read_parquet(..., chunk_size=...)` + `calc_data` | 大文件 bounded final | IO 分批，但最终语义是完整数据 | 一张最终结果表 |

简化理解：

- 想“像 DataFrame 一样算完”：用 `calc_data`；
- 想“数据一批批来，算子记住历史状态”：用 `calc_stream`。


## 11. ClickHouse 数据源

ClickHouse 读取走 `qust.datasource`：

```python
import qust.datasource as qds

ch = qds.ChConfig("host:9000", "username", "password", arrow_flight=False)
source = ch.as_datasource("database", "table", chunk_size=100_000, limit=1_000_000)
res = col.filter(col("trading_date") >= col.lit(date)).select(...).calc_data(source)
```

也可以直接传 query：

```python
source = qds.clickhouse(
    url="host:9000",
    database="future_market_data",
    username="bg",
    password="***",
    query="select * from gq_tick where trading_date = '2024-02-02'",
    chunk_size=100_000,
    arrow_flight=True,
)
```

下面的单元格只有在设置了环境变量后才会真实连接：

- `QUST_CH_URL`
- `QUST_CH_USER`
- `QUST_CH_PASSWORD`
- `QUST_CH_DB`
- `QUST_CH_TABLE`
- 可选：`QUST_CH_ARROW_FLIGHT=1`


In [79]:
required = ["QUST_CH_URL", "QUST_CH_USER", "QUST_CH_PASSWORD", "QUST_CH_DB", "QUST_CH_TABLE"]
missing = [name for name in required if not os.environ.get(name)]

if missing:
    display(pl.DataFrame({
        "status": ["skipped"],
        "reason": ["missing env: " + ", ".join(missing)],
    }))
else:
    ch = qds.ChConfig(
        os.environ["QUST_CH_URL"],
        os.environ["QUST_CH_USER"],
        os.environ["QUST_CH_PASSWORD"],
        arrow_flight=os.environ.get("QUST_CH_ARROW_FLIGHT", "0") in {"1", "true", "True"},
    )
    source = ch.as_datasource(
        os.environ["QUST_CH_DB"],
        os.environ["QUST_CH_TABLE"],
        chunk_size=100_000,
        limit=10_000,
    )
    display(col.all.select(col.all).calc_data(source).head(5))


status,reason
str,str
"""skipped""","""missing env: QUST_CH_URL, QUST…"


## 12. Debug：表达式显示、结构图和错误定位

基础算子学完以后，真正写策略和因子时最常遇到的问题不是“不会写某个函数”，而是：

- 表达式链太长，看不清当前到底在算哪一步；
- `select / with_cols / over / group_by / rolling` 嵌套后，错误只说某列不存在，不知道是哪一个子表达式触发；
- 想确认一条表达式的数据流方向：先算哪个 child，再进哪个 parent；
- 想逐步检查中间结果，而不是只看到最终报错。

qust 的 debug 工具主要解决这几件事：

| 工具 | 用途 |
| --- | --- |
| `expr.display()` / `repr(expr)` | 把表达式显示成尽量可复制到 Python 端运行的形式 |
| `expr.explain_text()` | 用文本查看表达式树，适合终端和日志 |
| `expr.explain()` | 用 monitor UI 查看表达式结构图 |
| `expr.calc_data_debug(data)` | 带 debug 上下文执行，报错时附加失败表达式、阶段和 schema |
| `expr.calc_data_debug(data, plot=True)` | 打开 debug plot UI，按数据流逐步查看执行节点、耗时和失败位置 |

下面用一个有 rolling、有 over、有派生列的表达式做示例。


### 12.1 `display()`：先把表达式看清楚

`display()` 的目标是让表达式尽量像 Python 代码一样可读。复杂表达式不一定 100% 还原你最初写代码时的换行和变量名，但应该能看出：

- 输入列是谁；
- 哪些地方是 `rolling` / `expanding` / `over`；
- 哪些地方做了 alias；
- 哪些地方是二元运算或横向组合。

这个方法适合在 notebook、日志、异常信息里快速确认“我手里的 expr 到底是什么”。


In [81]:
debug_data = data.head(200)

debug_expr = col(
    "ticker",
    "datetime",
    "open",
    "close",
    col("close").mean().rolling(20).alias("ma20"),
    ((col("close") / col("open")) - col.lit(1.0)).alias("bar_ret"),
).over("ticker")

print(debug_expr.display())


col(col("ticker"), col("datetime"), col("open"), col("close"), col("close").mean().rolling(20, 20).alias("ma20"), (col("close") / col("open")) - col.lit(1.0).alias("bar_ret")).over(col("ticker"))


### 12.2 `explain_text()` 和 `explain()`：看表达式树

`explain_text()` 是纯文本，适合复制到日志里。每一行大致包含：

- 节点 id：例如 `[0.1.2]`；
- 节点表达式；
- 节点类型：例如 `select`、`over`；
- 节点角色：例如 `root`、`group`、`callback`、`select[2]`。

`explain()` 会打开 monitor UI 版本的结构图。这个图只解释表达式结构，不执行数据；适合在表达式很长时看数据流方向。


In [82]:
print("\n".join(debug_expr.explain_text().splitlines()[:14]))


[0] col(col("ticker"), col("datetime"), col("open"), col("close"), col("close").mean().rolling(20, 20).alias("ma20"), (col("close") / col("open")) - col.lit(1.0).alias("bar_ret")).over(col("ticker")) (over) role=root
  [0.0] col("ticker") (ExprImpl) role=group
    [0.0.0] col("ticker") (ExprImpl) role=child[0]
  [0.1] col(col("ticker"), col("datetime"), col("open"), col("close"), col("close").mean().rolling(20, 20).alias("ma20"), (col("close") / col("open")) - col.lit(1.0).alias("bar_ret")) (select) role=callback
    [0.1.0] col("ticker") (ExprImpl) role=select[0]
    [0.1.1] col("datetime") (ExprImpl) role=select[1]
    [0.1.2] col("open") (ExprImpl) role=select[2]
    [0.1.3] col("close") (ExprImpl) role=select[3]
    [0.1.4] col("close").mean().rolling(20, 20).alias("ma20") (pipe) role=select[4]
      [0.1.4.0] col("close").mean().rolling(20, 20) (ExprImpl) role=child
        [0.1.4.0.0] col("close").mean() (pipe) role=child[0]
          [0.1.4.0.0.0] col("close") (ExprImpl) role=ch

In [83]:
explain_url = debug_expr.explain(open_in_jupyter=True, auto_open=False, height=620)
print(explain_url)


http://127.0.0.1:56715/index.html?config=18c6a77aa91bd6bf-13&v=1785243190611748561


### 12.3 `calc_data_debug(data)`：正常计算时结果不变

`calc_data_debug(data)` 会用 debug runtime 执行表达式。成功时，它返回的仍然是普通 `pl.DataFrame`，和 `calc_data(data)` 的结果口径一致。

区别在于：如果中间某个节点失败，debug runtime 会把失败节点、阶段、输入 schema 等信息附加到异常文本里。


In [84]:
normal_result = debug_expr.calc_data(debug_data)
debug_result = debug_expr.calc_data_debug(debug_data)

print("normal shape:", normal_result.shape)
print("debug shape:", debug_result.shape)
print("结果是否一致:", normal_result.equals(debug_result))
debug_result.head(8)


normal shape: (200, 6)
debug shape: (200, 6)
结果是否一致: True


ticker,datetime,open,close,ma20,bar_ret
str,datetime[ms],f64,f64,f64,f64
"""au""",2022-07-02 00:01:00,390.160004,390.119995,null,-0.000103
"""au""",2022-07-02 00:02:00.500,390.119995,390.140015,null,0.000051
"""au""",2022-07-02 00:03:00,390.140015,390.200012,null,0.000154
"""au""",2022-07-02 00:04:01,390.200012,390.160004,null,-0.000103
"""au""",2022-07-02 00:05:00.500,390.140015,390.100006,null,-0.000103
"""au""",2022-07-02 00:06:00.500,390.079987,390.140015,null,0.000154
"""au""",2022-07-02 00:07:00,390.119995,389.880005,null,-0.000615
"""au""",2022-07-02 00:08:03,389.859985,389.880005,null,0.000051


### 12.4 失败定位：找到真正出错的子表达式

下面故意构造一个错误：输入数据只保留 `open/high` 两列，但表达式里使用了不存在的 `low`。

普通报错往往只能告诉你“找不到 low”。debug 报错会额外给出：

- `failed expr`：最小失败表达式，这里应该定位到 `col("low")`；
- `debug stage`：失败发生在 schema 推导还是执行阶段；
- `debug input schema`：当前节点看到的输入列；
- `debug path`：失败节点所在路径。

在长策略里，这比只看到 `calc_data step=0` 有用得多。


In [85]:
bad_data = debug_data.select("open", "high")
bad_expr = col(
    "open",
    col("high").mean().rolling(3).alias("high_ma3"),
).select(
    col("open") + col("low")
)

try:
    bad_expr.calc_data_debug(bad_data)
except Exception as exc:
    debug_error_text = str(exc)
    interesting = []
    for line in debug_error_text.splitlines():
        if (
            "unable to find column" in line
            or line.startswith("failed expr:")
            or line.startswith("debug stage:")
            or line.startswith("debug input schema:")
            or line.startswith("debug path:")
        ):
            interesting.append(line)
    print("\n".join(interesting))


ColumnNotFound(ErrString("unable to find column \"low\"; valid columns: []\n\nResolved plan until failure:\n\n\t---> FAILED HERE RESOLVING 'select' <---\nDF []; PROJECT */0 COLUMNS"))
failed expr: col("low")
debug stage: fw
debug input schema: [open:Float64, high_ma3:Float64]
debug path: col("open") + col("low")
debug stage: fw: pipe.parent


### 12.5 `calc_data_debug(plot=True)`：逐算子查看执行结果

`plot=True` 会打开 debug plot UI。这个 UI 的重点不是画行情图，而是展示表达式图执行过程：

- 节点按表达式结构排列；
- 每一步有执行状态和耗时；
- 点击下一步可以按数据流方向逐步推进；
- 如果某个节点失败，对应节点会标红；
- 只有点击“保存当前结果”时，才会把当前结果保存到当前工作目录下的 parquet 文件。

下面对成功表达式打开 debug plot。这个输出是 notebook iframe，不是静态 PNG。


In [86]:
debug_plot_url = debug_expr.calc_data_debug(
    debug_data,
    plot=True,
    open_in_jupyter=True,
    auto_open=False,
    height=720,
)
print(debug_plot_url)


http://127.0.0.1:56715/index.html?config=18c6a77aaaaf76dc-2b&v=1785243190611748561


### 12.6 调试时的实用习惯

建议按这个顺序排查问题：

1. 先 `expr.display()`：确认表达式是不是你以为的那个表达式；
2. 再 `expr.explain_text()` 或 `expr.explain()`：确认 `select/with_cols/over/group_by` 的层级；
3. 小样本跑 `expr.calc_data_debug(data.head(...))`：优先定位 schema、列名、类型错误；
4. 对复杂表达式用 `plot=True`：按数据流逐步看节点，尤其适合定位长 pipeline 里的局部问题；
5. 如果是流式问题，用同一个 `runtime` 手动喂几个 batch，确认状态是否按预期保留。

debug 工具只改变“怎么报告和展示错误”，不应该改变普通 `calc_data` 的计算语义。
